<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/Donut/DocVQA/Creating_a_toy_DocVQA_dataset_for_Donut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating a toy DocVQA dataset for Donut

In this notebook, we'll show how to prepare a dataset for document visual question answering (DocVQA) for Donut.

The idea is to create a "ground_truth" column, which contains the ground truth JSON that the model needs to generate.

## Set-up environment

In [ ]:
!pip install -q datasets

## Load dataset

We'll create a minimal DocVQA toy dataset. Let's first load an existing one from the [🤗 hub](https://huggingface.co/).

In [ ]:
from datasets import load_dataset

# load a multilingual DocVQA dataset
dataset = load_dataset("olemeyer/docvqa-en-de-fr-es-it")

As can be seen, this dataset contains a train and test split.

In [ ]:
dataset

Let's look at an example:

In [ ]:
example = dataset['train'][0]
image = example['image']
width, height = image.size
image.resize((int(width*0.3), int(height*0.3)))

Each document has a corresponding question, in 5 languages:

In [ ]:
example['query']

Let's look at the corresponding answer(s):

In [ ]:
example['answers']

Note that, for a DocVQA dataset, it may happen that for a given question, multiple correct answers are possible (e.g. a short or longer version of a given name). This is why there's an "answers" column in the dataset, rather than just an "answer" column.

## Create toy dataset

We first just create a small subset of the dataset loaded above. We use the [select](https://huggingface.co/docs/datasets/v2.4.0/en/package_reference/main_classes#datasets.Dataset.select) method for that purpose.

In [ ]:
train_dataset = dataset["train"]

In [ ]:
train_dataset = train_dataset.select(range(1000))

In [ ]:
test_dataset = dataset["test"]

In [ ]:
test_dataset = test_dataset.select(range(200))

In [ ]:
from datasets import DatasetDict

toy_dataset = DatasetDict(
    {"train": train_dataset,
     "test": test_dataset,
})

In [ ]:
toy_dataset

We'll push this tiny subset to the hub.

In [ ]:
toy_dataset.push_to_hub("nielsr/docvqa_1200_examples")

## Add ground_truth for Donut

Donut requires you to add a column with some ground truth JSON/txt/JSON lines/whatever that you'd like the model to learn to generate. Below, we define a function that we'll apply on the entire dataset to add this column.

For DocVQA, the template looks like {gt_parses: [{"question" : {question_sentence}, "answer" : {answer_candidate_1}}, {"question" : {question_sentence}, "answer" : {answer_candidate_2}}, ...]}.

For example, {gt_parses: [{"question" : "what is the model name?", "answer" : "donut"}, {"question" : "what is the model name?", "answer" : "document understanding transformer"}]}.

The Donut authors also note: "in case your dataset has multiple answers, `gt_parses` should be a list of dictionaries, each containing a question-answer pair." In case your dataset only has single answers to each question, then you can create just a `gt_parse` rather than `gt_parses`.

In [ ]:
from datasets import load_dataset

toy_dataset = load_dataset("nielsr/docvqa_1200_examples_donut")

In [ ]:
import re

def add_ground_truth(examples):
  images = examples['image']
  queries = [query['en'] for query in examples['query']]
  answers = examples['answers']

  ground_truths = []
  for image, query, answers in zip(images, queries, answers):
    # we need to escape " characters appearing in the query and/or answer
    query = query.replace("\\", "") # this was just one corrupt example (index 91 of training set)
    query = re.sub(' +', ' ', query)
    query = query.replace('"', '\\"')
    # let's create the ground truth string
    ground_truth_example = '{"gt_parses": ['
    for idx, answer in enumerate(answers):
      answer = answer.replace('"', '\\"')
      ground_truth_example += '{"question" : "' + query + '", "answer" : "' + answer + '"}'
      # add comma
      if idx != len(answers) - 1:
        ground_truth_example += ', '
    ground_truth_example += ']}'
    ground_truths.append(ground_truth_example)
  
  examples['ground_truth'] = ground_truths
  
  return examples

toy_dataset = toy_dataset.map(add_ground_truth, batched=True)

It's important to verify that this added JSON can be properly read:

In [ ]:
import json

example = toy_dataset['train'][135]
json.loads(example['ground_truth'])

In [ ]:
example = toy_dataset['train'][58]
json.loads(example['ground_truth'])

In [ ]:
example = toy_dataset['train'][91]
json.loads(example['ground_truth'])

## Push to the hub

Finally, we push this dataset to the hub such that we easily reuse it later on.

In [ ]:
toy_dataset.push_to_hub("nielsr/docvqa_1200_examples_donut")